# Calibrate image-to-stage

Figure out how the camera's X and Y axes match the stage's X and Y axes.



## Step 1: Configure

Fill in the cell below:

- `session_id`: a name for this run (the date works well).
- `reference_objective`: which objective is on the microscope.
- `stage_move_um`: how far the stage should move during the test, in micrometers.

`SESSIONS_ROOT` is where this run's images and reports are saved. Edit that line
if you want a different location. The adopted calibration is published
separately as a dated snapshot under `C:\ProgramData\zmart-microscopy\...`.

Running this cell opens LAS X and prepares the session folder. Nothing is
acquired yet.

In [ ]:
import _bootstrap
from pathlib import Path
from navigator_expert.calibration.core import image_to_stage as wf

SESSIONS_ROOT = Path(
    r"C:\ProgramData\MinicondaZMB\home\t.de\navigator_expert_calibration\sessions"
)

session = wf.start_session(
    session_id="2026-05-22_scope_calibration",
    job_name="Overview",
    reference_objective="10x",
    stage_move_um=40.0,
    sessions_root=SESSIONS_ROOT,
)
print(session)

## Step 2: Measure

The microscope takes three pictures: one at the home position, one after
moving +X, one after moving +Y. The notebook then figures out which axes
match and saves the result.

The figure below shows four candidate orientations. The winner is named at
the top -- that tile's images line up cleanly; the other three show
magenta/green ghosting because they are wrong.


In [ ]:
session = wf.measure(session)
summary = wf.save_and_visualize(session)

## Step 3 (optional): Make this the active calibration

Run this cell only if the figure above looks right. It publishes the new
`calibration.json` as a dated **snapshot** under
`C:\ProgramData\zmart-microscopy\...` — carrying the existing single
`limits.json` (the function-keyed envelope + gate) and the
operator's `origin.json` forward, and archiving this executed notebook
alongside. Each snapshot dir holds exactly three files: `calibration.json`,
`limits.json`, `origin.json`. The previous snapshot is kept as history; the
driver reads the newest. (The physical stage envelope has its own factory,
`set_stage_limits.ipynb` under `limits/notebooks/`; a calibration adopt never
changes `limits.json`, only carries it forward.)


In [ ]:
from navigator_expert.calibration.core import adopt

adopt.adopt_calibration(
    session,
    staging_name="image_to_stage.json",
    notebook_paths=[_bootstrap.NOTEBOOKS_DIR / "calibrate_image_to_stage.ipynb"],
)